# Topic 11b: Softmax: From Two Classes to Many, by Hand, in scikit-learn, and in PyTorch

CSCI 3320 · AI Programming in Python · Fall 2026

This notebook is the code from the [Topic 11b lecture note](https://dongchul.kim/csci3320-fall2026/topic/10), every cell in the
same order, under the note's own section headings. The explanations are in the note: keep it
open beside this notebook. Run the cells from top to bottom (Runtime > Run all works too),
then change things and run them again.

It runs on a CPU; no GPU is needed. Numbers can differ a little from the note on a
different library version.

## Thirty seconds of scikit-learn, on the 569 patients

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()
X, y = data.data, 1 - data.target              # 1 = malignant, as in Topic 9b
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
scaler = StandardScaler().fit(X_train)         # mu and sigma from the training set only
Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)

model = LogisticRegression(max_iter=5000).fit(Z_train, y_train)
print(f"test accuracy: {model.score(Z_test, y_test):.4f}")

print("\nfirst three test patients: P(benign), P(malignant), and their sum")
for row in model.predict_proba(Z_test)[:3]:
    print(f"   {row[0]:.4f}   {row[1]:.4f}   {row.sum():.4f}")

## Now three classes: iris

In [ ]:
import numpy as np
from sklearn.datasets import load_iris

iris = load_iris()
X, y = iris.data, iris.target

print("X:", X.shape, "  y:", y.shape)
print("features:", iris.feature_names)
print("classes :", [str(c) for c in iris.target_names])
print("counts  :", np.bincount(y))

print("\none flower of each species:")
for i in [0, 50, 100]:
    print(f"  {X[i]}  ->  y = {y[i]}  ({iris.target_names[y[i]]})")

### The obvious patch, and why it is not enough

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)

answers = np.zeros((len(X_test), 3))
for k in range(3):                               # three yes/no models, three sigmoids
    model = LogisticRegression(max_iter=5000).fit(X_train, (y_train == k).astype(int))
    answers[:, k] = model.predict_proba(X_test)[:, 1]

print("P(setosa?)  P(versicolor?)  P(virginica?)    sum")
for row in answers[[3, 8, 12, 25]]:
    print(f"   {row[0]:.3f}         {row[1]:.3f}          {row[2]:.3f}      {row.sum():.3f}")

## Softmax: one score per class

### Step by step, with small numbers

In [ ]:
import numpy as np

z = np.array([2.0, 1.0, 0.1])       # one score per class: setosa, versicolor, virginica

e = np.exp(z)                        # step 1: exponentiate
p = e / e.sum()                      # step 2: divide by the total

print("scores         z =", z)
print("exp(z)           =", e.round(3))
print("sum of exp(z)    =", e.sum().round(3))
print("probabilities  p =", p.round(3))
print("p adds up to     =", p.sum().round(6))
print("predicted class  =", p.argmax())

### Why exponentiate?

In [ ]:
import numpy as np

def divide_by_sum(z):                # the obvious idea: no exp
    return z / z.sum()

def softmax(z):
    e = np.exp(z)
    return e / e.sum()

for z in [[2.0, 1.0, 0.1], [2.0, -1.0, 0.5], [1.0, -1.0, 0.0]]:
    z = np.array(z)
    print("scores", z)
    print("   divide by the sum:", divide_by_sum(z).round(3))
    print("   softmax          :", softmax(z).round(3))

## The loss: cross-entropy

In [ ]:
import numpy as np

def softmax(z):
    e = np.exp(z - z.max())      # subtracting the largest score keeps exp from overflowing; the probabilities are the same
    return e / e.sum()

p = softmax(np.array([2.0, 1.0, 0.1]))       # the model's answer for one flower
names = ["setosa", "versicolor", "virginica"]
print("model says:", p.round(3))

for true_class in range(3):
    y = np.eye(3)[true_class]                # one-hot, e.g. versicolor -> [0, 1, 0]
    loss = -np.sum(y * np.log(p))            # only the true class's term survives
    print(f"truly {names[true_class]:<10}  y = {y}   loss = -log({p[true_class]:.4f}) = {loss:.3f}")

# two classes: the same formula is Topic 9a's log loss
p1, y1 = 0.9, 1
print(f"\n9a log loss   : {-(y1 * np.log(p1) + (1 - y1) * np.log(1 - p1)):.4f}")
print(f"cross-entropy : {-np.sum(np.array([1 - y1, y1]) * np.log(np.array([1 - p1, p1]))):.4f}")

## Softmax regression by hand, on iris

### The loop

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)   # 10 test flowers per species

scaler = StandardScaler().fit(X_train)                 # mu and sigma from TRAIN only
Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)

def softmax(S):                                        # one row of scores per flower
    E = np.exp(S - S.max(axis=1, keepdims=True))       # subtract each row's largest score so exp cannot overflow (same probabilities)
    return E / E.sum(axis=1, keepdims=True)

Y_train = np.eye(3)[y_train]                           # one-hot: 2 -> [0, 0, 1]
n, m = Z_train.shape                                   # 120 flowers, 4 features

W, b = np.zeros((m, 3)), np.zeros(3)                   # one weight column per class
alpha = 0.5
print(" step    loss    train accuracy")
for step in range(1001):
    P = softmax(Z_train @ W + b)                       # (120, 3) probabilities

    if step % 100 == 0:
        loss = -np.mean(np.sum(Y_train * np.log(P), axis=1))
        acc  = (P.argmax(axis=1) == y_train).mean()
        print(f"{step:5d}   {loss:.4f}    {acc:.4f}")

    W -= alpha * (Z_train.T @ (P - Y_train) / n)       # the 9a step, 3 columns at once
    b -= alpha * np.mean(P - Y_train, axis=0)

pred = softmax(Z_test @ W + b).argmax(axis=1)
print(f"\ntest accuracy: {(pred == y_test).mean():.4f}  ({(pred == y_test).sum()} of {len(y_test)})")
print("\nW, one column per class:\n", W.round(3))
print("b:", b.round(3))

## The same model in scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)
scaler = StandardScaler().fit(X_train)
Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)

# theirs: three classes, so LogisticRegression uses softmax on its own
model = LogisticRegression(max_iter=5000).fit(Z_train, y_train)
theirs = model.predict_proba(Z_test)
print(f"scikit-learn test accuracy: {model.score(Z_test, y_test):.4f}")
print("coef_ shape:", model.coef_.shape)
print("first three rows of predict_proba:\n", theirs[:3].round(3))
print("their sums:", theirs[:3].sum(axis=1).round(6))

# ours: the loop from above, unchanged
def softmax(S):
    E = np.exp(S - S.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)

Y_train, n = np.eye(3)[y_train], len(y_train)
W, b = np.zeros((4, 3)), np.zeros(3)
for step in range(1001):
    P = softmax(Z_train @ W + b)
    W -= 0.5 * (Z_train.T @ (P - Y_train) / n)
    b -= 0.5 * np.mean(P - Y_train, axis=0)
ours = softmax(Z_test @ W + b)

diff = np.where(ours.argmax(axis=1) != theirs.argmax(axis=1))[0]
print("\ntest flowers where the two models disagree:", len(diff), "of", len(y_test))
for i in diff:
    print(f"  flower {i}, truly class {y_test[i]}")
    print("  ours         ", ours[i].round(3))
    print("  scikit-learn ", theirs[i].round(3))
print(f"largest weight: ours {np.abs(W).max():.2f}, scikit-learn {np.abs(model.coef_).max():.2f}")

loose = LogisticRegression(C=1e6, max_iter=5000).fit(Z_train, y_train)
print("disagreements with C=1e6:", (loose.predict(Z_test) != ours.argmax(axis=1)).sum())

## Meet PyTorch

### Tensors, and moving between PyTorch and NumPy

In [ ]:
import numpy as np
import torch

a = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(a)
print("shape:", a.shape, "  dtype:", a.dtype, "  device:", a.device)
print("GPU available on this laptop?", torch.cuda.is_available())

t = torch.from_numpy(np.array([[1.0, 2.0], [3.0, 4.0]]))    # NumPy -> PyTorch
print("\nfrom NumPy:", t.dtype)
print("back to NumPy:", type(t.numpy()))                     # PyTorch -> NumPy

w = torch.zeros(2, 1)                                        # float32, PyTorch's default
try:
    t @ w
except RuntimeError as err:
    print("\nfloat64 @ float32  ->  RuntimeError:", err)

### The operations you already know

In [ ]:
import torch

X = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])     # 2 rows, 3 columns
W = torch.tensor([[1.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 1.0]])          # 3 rows, 2 columns

print("X @ W =\n", X @ W)               # (2, 3) @ (3, 2) -> (2, 2), exactly as in NumPy
print("X.T shape:", X.T.shape)
print("X.sum(dim=0) =", X.sum(dim=0), "   (dim is NumPy's axis)")

z = torch.tensor([2.0, 1.0, 0.1])
print("\ntorch.softmax(z, dim=0) =", torch.softmax(z, dim=0))
print("argmax:", z.argmax().item())

## Iris again, in PyTorch

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)
scaler = StandardScaler().fit(X_train)
Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)
Y_train = np.eye(3)[y_train]
n, alpha = len(y_train), 0.5
W0, b0 = np.zeros((4, 3)), np.zeros(3)                  # ONE starting point for both

# ---- NumPy: the loop from the by-hand section ----
def softmax(S):
    E = np.exp(S - S.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)

W_np, b_np = W0.copy(), b0.copy()
for step in range(1001):
    P = softmax(Z_train @ W_np + b_np)
    grad_W = Z_train.T @ (P - Y_train) / n              # our formula
    if step == 0:
        first_grad = grad_W
    W_np -= alpha * grad_W
    b_np -= alpha * np.mean(P - Y_train, axis=0)

# ---- PyTorch: backward() finds the gradient, the optimizer takes the step ----
Xt, yt = torch.from_numpy(Z_train), torch.from_numpy(y_train)
W = torch.tensor(W0, requires_grad=True)                 # the same start, float64
b = torch.tensor(b0, requires_grad=True)
optimizer = torch.optim.SGD([W, b], lr=alpha)            # plain gradient descent

for step in range(1001):
    loss = F.cross_entropy(Xt @ W + b, yt)               # scores in, loss out
    optimizer.zero_grad()                                # clear the last step's gradient
    loss.backward()                                      # fills in W.grad and b.grad
    if step == 0:
        print("loss at the start:", loss.item())
        print("W.grad at the start:\n", W.grad)
        print("largest |W.grad - our formula|:", np.abs(W.grad.numpy() - first_grad).max())
    optimizer.step()                                     # W -= alpha * W.grad, b likewise

print("\nafter training, largest |W_numpy - W_torch|:", np.abs(W_np - W.detach().numpy()).max())
print("after training, largest |b_numpy - b_torch|:", np.abs(b_np - b.detach().numpy()).max())

pred_np = (Z_test @ W_np + b_np).argmax(axis=1)          # the largest score wins
pred_t = (torch.from_numpy(Z_test) @ W + b).argmax(dim=1).numpy()
print(f"\ntest accuracy: NumPy {(pred_np == y_test).mean():.4f}   PyTorch {(pred_t == y_test).mean():.4f}")
print("test flowers where they disagree:", (pred_np != pred_t).sum(), "of", len(y_test))

## From three classes to ten: handwritten digits

In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

X, y = load_digits(return_X_y=True)                     # 1797 images, 64 pixels each
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)
print("pixels that are 0 in every training image:", (X_train.std(axis=0) == 0).sum())

scaler = StandardScaler().fit(X_train)                  # leaves those constant pixels alone
Xt = torch.tensor(scaler.transform(X_train), dtype=torch.float32)   # float32 this time
Xv = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
yt = torch.from_numpy(y_train)

model = nn.Linear(64, 10)                               # a W and a b in one object
nn.init.zeros_(model.weight)                            # start from zeros, as on iris
nn.init.zeros_(model.bias)
loss_fn = nn.CrossEntropyLoss()                         # softmax + cross-entropy, from scores
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
print("weight:", tuple(model.weight.shape), "  bias:", tuple(model.bias.shape))

print("\n step    loss    train accuracy")
for step in range(1001):
    scores = model(Xt)                                  # Xt @ W + b: (1437, 10) scores
    loss = loss_fn(scores, yt)
    if step % 200 == 0:
        acc = (scores.argmax(dim=1) == yt).float().mean()
        print(f"{step:5d}   {loss.item():.4f}    {acc:.4f}")
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

pred = model(Xv).argmax(dim=1).numpy()
print(f"\ntest accuracy: {(pred == y_test).mean():.4f}  ({(pred == y_test).sum()} of {len(y_test)})")
print("\nconfusion matrix, rows = true digit, columns = predicted digit")
print(confusion_matrix(y_test, pred))

Back to the [Topic 11b lecture note](https://dongchul.kim/csci3320-fall2026/topic/10).